In [4]:
import pandas as pd
import numpy as np
import os

# Data Cleaning

## Basic Cleaning
Here I'm just cleaning up the data to be better formatted to work with. This is more focused on separating columns with multiple values into multiple columns not just a single one, renaming columns where I feel necessary, dropping some columns, fixing the date time format, and anything else I deem necessary.

### Global Variable

In [5]:
month_choices = [9, 10, 11, 12, 1, 2]

Chaning the working directory so I don't have to hard code long path names

In [ ]:
print(f'Current: {os.getcwd()}')
os.chdir('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/htmlparse')
print(f'Current: {os.getcwd()}')

Current: /Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/htmlparse


### Team Statistics

In [ ]:
team_stats = pd.read_csv('team_stats.csv')

team_stats = team_stats.drop(columns='Unnamed: 0')
team_stats = team_stats.rename(columns={'Cmp-Att-Yd-TD-INT':'ManyStats'})
team_stats[['Completion', 'Attempts', 'Yards', 'Touchdowns', 'Interceptions']] = team_stats.ManyStats.str.split('-', expand=True)
team_stats = team_stats.drop(columns='ManyStats')

team_stats.columns = team_stats.columns.str.replace(' ', '_')
team_stats.columns = team_stats.columns.str.replace('.', '_')
team_stats.columns = team_stats.columns.str.replace('-', '_')

team_stats[['Success4thDown', 'Failed4thDown']] = team_stats.Fourth_Down_Conv_.str.split('-', expand=True)
team_stats[['Fumbles', 'FumblesLost']] = team_stats.Fumbles_Lost.str.split('-', expand=True)
team_stats[['Penalties', 'PenaltyYards']] = team_stats.Penalties_Yards.str.split('-', expand=True)
#team_stats[['RushAttempts', 'RushYards', 'RushTDs']] = team_stats.Rush_Yds_TDs.str.split('-', expand=True)
team_stats[['Sacked', 'SackYards']] = team_stats.Sacked_Yards.str.split('-', expand=True)
team_stats[['Success3rdDown', 'Failed3rdDown']] = team_stats.Third_Down_Conv_.str.split('-', expand=True)
team_stats[['TimeMinutes', 'TimeSeconds']] = team_stats.Time_of_Possession.str.split(':', expand=True)

team_stats = team_stats.drop(columns={'Fourth_Down_Conv_', 'Fumbles_Lost', 'Penalties_Yards', 'Sacked_Yards', 'Third_Down_Conv_', 'Time_of_Possession'})

team_stats['Rush_Yds_TDs'].str.count('-').value_counts()

team_stats.loc[2083]

team_stats.at[2083,'Rush_Yds_TDs'] = '8-neg1-0'

team_stats[['RushAttempts', 'RushYards', 'RushTDs']] = team_stats.Rush_Yds_TDs.str.split('-', expand=True)
team_stats = team_stats.drop(columns='Rush_Yds_TDs')

team_stats.at[2083, "RushYards"] = -1

skip_cols = ['team_name', 'date']
cols_to_change = [col for col in team_stats.columns if col not in skip_cols]
team_stats[cols_to_change] = team_stats[cols_to_change].astype(int)

team_stats['CompletionPct'] = round(100 * (team_stats['Completion'] / team_stats['Attempts']), 1)
team_stats['PctYardsPass'] = round(100 * (team_stats['Yards'] / team_stats['Total_Yards']), 1)
team_stats['PctYardsRush'] = round(100 * (team_stats['RushYards'] / team_stats['Total_Yards']), 1)
team_stats['SuccessRate4thDown'] = round(
    100 * (team_stats['Success4thDown'] / (team_stats['Success4thDown'] + team_stats['Failed4thDown'])), 1)
team_stats['SuccessRate3rdDown'] = round(
    100 * (team_stats['Success3rdDown'] / (team_stats['Success3rdDown'] + team_stats['Failed3rdDown'])), 1)
team_stats['PossessionTime'] = (60 * team_stats['TimeMinutes']) + team_stats['TimeSeconds']

team_stats[['DayofWeek', 'Month', 'DayofMonth', 'Year']] = team_stats.date.str.split(' ', expand=True)
team_stats['DayofMonth'] = team_stats['DayofMonth'].str.rstrip(',')

game_months = [
    (team_stats['Month'] == 'Sep'),
    (team_stats['Month'] == 'Oct'),
    (team_stats['Month'] == 'Nov'),
    (team_stats['Month'] == 'Dec'),
    (team_stats['Month'] == 'Jan'),
    (team_stats['Month'] == 'Feb')
]
month_choices = [9, 10, 11, 12, 1, 2]

team_stats['Month'] = np.select(game_months, month_choices, default=False)
team_stats['DayofMonth'] = team_stats['DayofMonth'].astype(int)
team_stats['Year'] = team_stats['Year'].astype(int)

date_mapping = {
    'Year':'year',
    'DayofMonth':'day',
    'Month':'month'}

team_stats = team_stats.rename(mapper=date_mapping, axis=1)
team_stats['gamedate'] = pd.to_datetime(team_stats[['year', 'month', 'day']])

team_stats = team_stats.drop(columns={'date','Completion','Success4thDown','Failed4thDown','Success3rdDown','Failed3rdDown', 'day'})

### Offensive Player Statistics

In [ ]:
player_offense = pd.read_csv('player_offense.csv')

new_offense_cols = ['randnumber','playername','team','completions','passattempts',
                    'passingyards','passingtds','interceptions','sackstaken','yardslostbysack',
                    'longestcompletion','passrating','rushingattempts','rushyards','rushtds',
                    'longestrush','targets','receptions','receivingyards','receivingtds',
                    'longestreception','fumbles','fumbleslost','gamedate']

player_offense.columns = new_offense_cols

realplayer = player_offense['playername'] != "Player"
playerhere = player_offense['playername'].notna()
player_offense = player_offense[realplayer]
player_offense = player_offense[playerhere]

player_offense[['dayofweek','month','day','year']] = player_offense.gamedate.str.split(' ', expand=True)
player_offense = player_offense.drop(columns='gamedate')
player_offense['day'] = player_offense['day'].str.rstrip(',')

game_months = [
    (player_offense['month'] == 'Sep'),
    (player_offense['month'] == 'Oct'),
    (player_offense['month'] == 'Nov'),
    (player_offense['month'] == 'Dec'),
    (player_offense['month'] == 'Jan'),
    (player_offense['month'] == 'Feb')
]

player_offense['month'] = np.select(game_months, month_choices, default=False)
player_offense['gamedate'] = pd.to_datetime(player_offense[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in player_offense.columns if col not in skip_cols]
player_offense[cols_to_change] = player_offense[cols_to_change].astype(float)

player_offense['completionpct'] = round(100 * (player_offense['completions'] / player_offense['passattempts']), 1)
player_offense['td2int'] = round(player_offense['passingtds'] / player_offense['interceptions'], 3)
player_offense['yardsperrush'] = round(player_offense['rushyards'] / player_offense['rushingattempts'], 1)

player_offense = player_offense.drop(columns={'randnumber','day','month','year'})

/var/folders/xv/_79n8tfd6mx4j2z8m6rstygr0000gn/T/ipykernel_7114/3584193664.py:14: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  player_offense = player_offense[playerhere]


In [6]:
qbs = player_offense[player_offense['passattempts'] >= 10]
rbs = player_offense[(player_offense['rushingattempts'] >= 10) & (player_offense['passattempts'] <= 2)]
wrs = player_offense[player_offense['targets'] >= 2]

### Defensive Player Statistics

In [ ]:
player_defense = pd.read_csv('player_defense.csv')

new_defense_cols = ['randnum','playername','team','interceptions','intrtnyards','int2td','intlongestyards',
                    'passdeflections','sacks','soloassisttackles','solotackles','assisttackles','tackelsforloss',
                    'qbhits','fumblerecovery','fumblereturnyards','fumble2td','forcedfumble','gamedate']
player_defense.columns = new_defense_cols

player_defense = player_defense[(player_defense['playername'] != 'Player') & player_defense['playername'].notna()]

player_defense[['dayofweek','month','day','year']] = player_defense.gamedate.str.split(' ', expand=True)
player_defense = player_defense.drop(columns='gamedate')
player_defense['day'] = player_defense['day'].str.rstrip(',')

game_months = [
    (player_defense['month'] == 'Sep'),
    (player_defense['month'] == 'Oct'),
    (player_defense['month'] == 'Nov'),
    (player_defense['month'] == 'Dec'),
    (player_defense['month'] == 'Jan'),
    (player_defense['month'] == 'Feb')
]

player_defense['month'] = np.select(game_months, month_choices, default=False)
player_defense['gamedate'] = pd.to_datetime(player_defense[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in player_defense.columns if col not in skip_cols]
player_defense[cols_to_change] = player_defense[cols_to_change].astype(float)

player_defense = player_defense.drop(columns={'randnum','day','month','year'})

### Advanced Passing Statistics

In [ ]:
advanced_passing = pd.read_csv('passing_advanced.csv')
advanced_passing = advanced_passing.drop(columns={'Unnamed: 0','Cmp','1D'})
new_advpass_cols = ['playername','team','attempts','passingyards','firstdownsperpassplay',
                    'intendairyards','intendairyardsperpassatt','completedairyards',
                    'completedairyardspercomp','completedairyardsperpassatt',
                    'yardsaftercatch','yacpercompletion','drops','percentdrops',
                    'badthrows','badthrowspercent','sackstaken','timesblitzed',
                    'timeshurried','hitstaken','timespressured','percentdbspressured',
                    'scrambles','yardsperscramble','date']

advanced_passing.columns = new_advpass_cols

advanced_passing = advanced_passing[advanced_passing['playername'] != 'Player']

advanced_passing['percentdbspressured'] = advanced_passing['percentdbspressured'].str.rstrip('%')
advanced_passing['badthrowspercent'] = advanced_passing['badthrowspercent'].str.rstrip('%')
advanced_passing['percentdrops'] = advanced_passing['percentdrops'].str.rstrip('%')

advanced_passing[['dayofweek','month','day','year']] = advanced_passing.date.str.split(' ', expand=True)
advanced_passing = advanced_passing.drop(columns='date')
advanced_passing['day'] = advanced_passing['day'].str.rstrip(',')

game_months = [
    (advanced_passing['month'] == 'Sep'),
    (advanced_passing['month'] == 'Oct'),
    (advanced_passing['month'] == 'Nov'),
    (advanced_passing['month'] == 'Dec'),
    (advanced_passing['month'] == 'Jan'),
    (advanced_passing['month'] == 'Feb')
]

advanced_passing['month'] = np.select(game_months, month_choices, default=False)
advanced_passing['gamedate'] = pd.to_datetime(advanced_passing[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in advanced_passing.columns if col not in skip_cols]
advanced_passing[cols_to_change] = advanced_passing[cols_to_change].astype(float)

advanced_passing = advanced_passing.drop(columns={'day','month','year'})

### Advanced Receiving Statistics

In [ ]:
advanced_receiving = pd.read_csv('receiving_advanced.csv')
advanced_receiving = advanced_receiving.drop(columns={'Unnamed: 0','Rec/Br'})
new_advrec_cols = ['playername','team','targets','receptions','yards','touchdowns',
                   'firstdownsreceiving','yardsbeforecatch','yardsbeforecatchper',
                   'yardsaftercatch','yardsaftercatchper','avgdeptoftarget',
                   'brokentackles','drops','droprate','intsontargets',
                   'passrtgontargets','date']

advanced_receiving.columns = new_advrec_cols

advanced_receiving = advanced_receiving[advanced_receiving['playername'] != 'Player']

advanced_receiving[['dayofweek','month','day','year']] = advanced_receiving.date.str.split(' ', expand=True)
advanced_receiving = advanced_receiving.drop(columns='date')
advanced_receiving['day'] = advanced_receiving['day'].str.rstrip(',')

game_months = [
    (advanced_receiving['month'] == 'Sep'),
    (advanced_receiving['month'] == 'Oct'),
    (advanced_receiving['month'] == 'Nov'),
    (advanced_receiving['month'] == 'Dec'),
    (advanced_receiving['month'] == 'Jan'),
    (advanced_receiving['month'] == 'Feb')
]

advanced_receiving['month'] = np.select(game_months, month_choices, default=False)
advanced_receiving['gamedate'] = pd.to_datetime(advanced_receiving[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in advanced_receiving.columns if col not in skip_cols]
advanced_receiving[cols_to_change] = advanced_receiving[cols_to_change].astype(float)

advanced_receiving = advanced_receiving.drop(columns={'day','month','year'})

### Advanced Rushing

In [ ]:
advanced_rushing = pd.read_csv('rushing_advanced.csv')
advanced_rushing = advanced_rushing.drop(columns={'Unnamed: 0'})
new_advrush_cols = ['playername','team','rushes','attempts','touchdowns','firstdownsrushing',
                    'yardsbeforecontact','yardsbeforecontactper','yardsaftercontact',
                    'yardsaftercontactper','brokentackles','brokentacklesper','date']

advanced_rushing.columns = new_advrush_cols

advanced_rushing = advanced_rushing[advanced_rushing['playername'] != 'Player']

advanced_rushing[['dayofweek','month','day','year']] = advanced_rushing.date.str.split(' ', expand=True)
advanced_rushing = advanced_rushing.drop(columns='date')
advanced_rushing['day'] = advanced_rushing['day'].str.rstrip(',')

game_months = [
    (advanced_rushing['month'] == 'Sep'),
    (advanced_rushing['month'] == 'Oct'),
    (advanced_rushing['month'] == 'Nov'),
    (advanced_rushing['month'] == 'Dec'),
    (advanced_rushing['month'] == 'Jan'),
    (advanced_rushing['month'] == 'Feb')
]

advanced_rushing['month'] = np.select(game_months, month_choices, default=False)
advanced_rushing['gamedate'] = pd.to_datetime(advanced_rushing[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in advanced_rushing.columns if col not in skip_cols]
advanced_rushing[cols_to_change] = advanced_rushing[cols_to_change].astype(float)

advanced_rushing = advanced_rushing.drop(columns={'day','month','year'})

### Advanced Defensive Statistics

In [ ]:
advanced_defense = pd.read_csv('defense_advanced.csv')
new_advdef_cols = ['randnum','playername','team','interceptions','deftargets','completions',
                   'cmppctincoverage','yardsallowed','yardsallowedpercmp',
                   'yardspertarget','tdsallowed','passrtgallowed',
                   'agvdepthoftargetcvg','airyardsoncmps','yardsaftercatch',
                   'blitzes','qbhurries','qbknockdown','sacks','pressures',
                   'soloassisttackles','missedtackles','missedtacklepct','date']
advanced_defense.columns = new_advdef_cols

advanced_defense = advanced_defense.drop(columns={'randnum', 'completions','airyardsoncmps'})

advanced_defense = advanced_defense[advanced_defense['playername'] != 'Player']

advanced_defense['cmppctincoverage'] = advanced_defense['cmppctincoverage'].str.rstrip('%')
advanced_defense['missedtacklepct'] = advanced_defense['missedtacklepct'].str.rstrip('%')

advanced_defense[['dayofweek','month','day','year']] = advanced_defense.date.str.split(' ', expand=True)
advanced_defense = advanced_defense.drop(columns='date')
advanced_defense['day'] = advanced_defense['day'].str.rstrip(',')

game_months = [
    (advanced_defense['month'] == 'Sep'),
    (advanced_defense['month'] == 'Oct'),
    (advanced_defense['month'] == 'Nov'),
    (advanced_defense['month'] == 'Dec'),
    (advanced_defense['month'] == 'Jan'),
    (advanced_defense['month'] == 'Feb')
]

advanced_defense['month'] = np.select(game_months, month_choices, default=False)
advanced_defense['gamedate'] = pd.to_datetime(advanced_defense[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in advanced_defense.columns if col not in skip_cols]
advanced_defense[cols_to_change] = advanced_defense[cols_to_change].astype(float)

advanced_defense = advanced_defense.drop(columns={'day','month','year'})

#### Export to CSV
Ensuring that I don't have to rerun all these chunks of code each time

In [ ]:
print(f'Current: {os.getcwd()}')
os.chdir('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/cleaned')

In [ ]:
#team_stats.to_csv('team_stats.csv')
#qbs.to_csv('qbs.csv')
#rbs.to_csv('rbs.csv')
#wrs.to_csv('wrs.csv')
#player_defense.to_csv('player_defense.csv')
#advanced_passing.to_csv('adv_pass.csv')
#advanced_receiving.to_csv('adv_pass.csv')
#advanced_rushing.to_csv('adv_rush.csv')
#advanced_defense.to_csv('adv_def.csv')

## Pre-Analysis Cleaning

In [ ]:
print(f'Current: {os.getcwd()}')
os.chdir('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/cleaned')
print(f'Current: {os.getcwd()}')

Current: /Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/cleaned


In [72]:
team_stats = pd.read_csv('team_stats.csv')
team_stats = team_stats.drop(columns={'Unnamed: 0','TimeMinutes','TimeSeconds'})
team_stats['Yards_Per_Penalty'] = round(team_stats['PenaltyYards'] / team_stats['Penalties'], 1)
team_stats = team_stats[team_stats['year'] != 2025]

In [ ]:
avg_stats = team_stats.groupby(
    ['team_name','year']).agg(
        {'First_Downs':'mean', 'Turnovers':'mean', 'Attempts':'mean', 'Yards':'mean',
         'Interceptions':'mean', 'RushAttempts':'mean', 'RushYards':'mean',
         'CompletionPct':'mean', 'PctYardsPass':'mean', 'PctYardsRush':'mean',
         'SuccessRate4thDown':'mean', 'SuccessRate3rdDown':'mean', 'PossessionTime':'mean',
         'Yards_Per_Penalty':'mean', 'Penalties':'mean'
            }).round(1)

avg_stats = avg_stats.add_suffix('_avg')

back_2_og = pd.merge(left=team_stats, right=avg_stats, left_on=['team_name','year'], right_on=['team_name','year'], how='left')

back_2_og = back_2_og[['team_name'] + [col for col in back_2_og.columns if col != 'team_name']]
back_2_og = back_2_og[['gamedate'] + [col for col in back_2_og.columns if col != 'gamedate']]
back_2_og = back_2_og.drop(columns={'DayofWeek','month'})